In [0]:
from pyspark.sql.functions import regexp_extract, col
import os

VOLUME_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/set/" 

FOLDERS_TO_PROCESS = ["depression"]

def create_raw_sessions_table(base_path, folders, table_name="anima.raw_eye_tracking"):
    """
    Ingests session CSVs, adds the filename as 'session_id', and saves to a Delta table.
    """
    full_paths = [f"{base_path}{folder}/*.csv" for folder in folders]
    
    print(f"Reading data from: {full_paths}")

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(full_paths)

    df_with_id = df.withColumn(
        "session_id", 
        regexp_extract(col("_metadata.file_path"), r"([^/]+)\.csv$", 1)
    )

    df_clean = df_with_id.filter(col("session_id") != "")

    print("Writing to Delta Table...")
    
    df_clean.write.format("delta").mode("overwrite").saveAsTable(table_name)
    
    print(f"Success! Table '{table_name}' created.")
    return df_clean

df_sessions = create_raw_sessions_table(VOLUME_PATH, FOLDERS_TO_PROCESS)

In [0]:
%sql
select distinct session_id from anima.raw_eye_tracking

In [0]:
from pyspark.sql.functions import col, count, desc

FORMS_FILE_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/forms.csv"
TABLE_NAME = "anima.forms"

print(f"Reading forms data from: {FORMS_FILE_PATH}")

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(FORMS_FILE_PATH)

df_raw.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)
print(f"Success! Table '{TABLE_NAME}' created.")

df_forms = spark.table(TABLE_NAME)

total_users = df_forms.select("uid").distinct().count()
total_sessions = df_forms.count()

print(f"\n--- Statistics ---")
print(f"Total Unique Users: {total_users}")
print(f"Total Sessions: {total_sessions}")

user_session_counts = df_forms.groupBy("uid") \
    .agg(count("sid").alias("session_count")) \
    .orderBy(desc("session_count"))

print("\nTop 10 Users (Candidates for Longitudinal Study):")
display(user_session_counts.limit(10))

distribution = user_session_counts.groupBy("session_count") \
    .count() \
    .withColumnRenamed("count", "num_users") \
    .orderBy("session_count")

print("\nHow many users have X sessions?")
display(distribution)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

pdf = distribution.toPandas()

pdf = pdf.sort_values('session_count')

plt.figure(figsize=(14, 8))
bars = plt.bar(pdf['session_count'], pdf['num_users'], color='#4c72b0', edgecolor='white')

for bar in bars:
    height = bar.get_height()
    if height > 0:
        plt.text(
            bar.get_x() + bar.get_width()/2., 
            height + 10,
            f'{int(height)}', 
            ha='center', 
            va='bottom', 
            fontsize=9, 
            rotation=90 if height > 100 else 0
        )

plt.xlabel('Number of Sessions per User', fontsize=12)
plt.ylabel('Number of Users (Count)', fontsize=12)
plt.title('Distribution: How many users have X sessions?', fontsize=14)

plt.xticks(pdf['session_count'], rotation=90, fontsize=8)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()